In [ ]:
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles.fills import PatternFill
import os

In [ ]:
# Customizable Variables

spreadsheet_file_name = 'Surcharges_8-16Sep.xlsx'
highlight_color = 'FFFF00'
new_file_name = f'{os.getcwd()}/highlighted/hightlighted-{spreadsheet_file_name}'

# Customize billable days
date_col = 'F H J L T V X Z AB AH AJ'
num_days_col = 'AL'

# Set targeted timing
time_format = '%H:%M'
early_time = '10:30'
late_time = '14:30'

In [ ]:
date_col = date_col.split()
date_col

In [ ]:
# Functions
def get_column_name(index):
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    column_name = ''
    while index >= 0:
        column_name = alphabet[(index % len(alphabet))] + column_name
        index = (index // len(alphabet) -1)
    return column_name
def get_column_index(name):
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    length = len(name)
    i = length
    value = 0
    for letter in name:
        i = i - 1
        value = value + (alphabet.index(letter) + 1) * pow(26, i)
        
    return value

early_target = datetime.strptime(early_time, time_format)
late_target = datetime.strptime(late_time, time_format)

# Read the file
df = pd.read_excel(spreadsheet_file_name, header=None)
workbook = load_workbook(spreadsheet_file_name)
sheet = workbook.active
column_list = []
for i in range(df.shape[1]):
    column_list.append(get_column_name(i))
df.columns = column_list

In [ ]:
# Count the number of students

total_students = df['A'].nunique()

print(f"Number of students: {total_students}")

In [ ]:

def highlight_cell(r, c):
    print(f'highlighting row: {r + 1}, col {get_column_name(c)} {c}')
    cell = sheet.cell(row=(r + 1), column=c)
    highlight = PatternFill(fill_type="solid", fgColor=highlight_color)
    cell.fill = highlight

# Loop through each student
student_list = df.groupby(df['A'])
for index, student in student_list:
    name = student.iloc[0]['B']
    sign_in_row = student.index.min()
    sign_out_row = sign_in_row + 1
    for column in date_col:
        if pd.isna(df.at[sign_in_row, column]) == False:
            sign_in_time = datetime.strptime(str(df.at[sign_in_row, column]), time_format)
            sign_out_time = datetime.strptime(str(df.at[sign_out_row, column]), time_format)
            #print(f"Name: {name}, Date: {df.at[8, column]}, index: {int(index)}, Sign in: {sign_in_time}, Sign out: {sign_out_time},")
            if sign_in_time < early_target and sign_out_time > late_target: 
                highlight_cell(sign_in_row, get_column_index(column))
                highlight_cell(sign_out_row, get_column_index(column))
                #Sprint(f"Name: {name}, Date: {df.at[8, column]}, Sign in: {sign_in_time.time()}, Sign out: {sign_out_time.time()},")
workbook.save(new_file_name)

In [ ]:
# Enter number of days
for index, student in student_list:
    student_name = student.iloc[0]['B']
    count = 0
    row = student.index.min() + 1
    for col in date_col:
        cell = sheet.cell(row=row, column=get_column_index(col))
        cell_color = cell.fill.fgColor.rgb
        if(cell.fill and cell_color == f'00{highlight_color}'):
            print(f'row {row} col {col}')
            count += 1
    if(count > 0):
        cell = sheet.cell(row=row, column=get_column_index(num_days_col))
        cell.value = count
        print(f'{student_name} has {count} days')

workbook.save(new_file_name)